# Extração dos assuntos processuais do CNJ
Execute primeiro `%pip install requests==2.32.3` no mesmo kernel, se necessário. A célula abaixo gera o CSV na pasta atual.


In [ ]:
"""Extrai a árvore de assuntos processuais do CNJ para um CSV único.

Fonte: WebService público do SGT, consultado pela mesma base da página
https://www.cnj.jus.br/sgt/consulta_publica_assuntos.php.

As categorias raiz foram conferidas com a versão exibida em 12/09/2026.
"""

from __future__ import annotations

import csv
import hashlib
import json
import logging
import os
import tempfile
import threading
import xml.etree.ElementTree as ET
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Protocol


# ================================================================
# 1. CONFIGURAÇÕES ALTERÁVEIS
# ================================================================

WSDL_URL = "https://www.cnj.jus.br/sgt/sgt_ws.php?wsdl"
OUTPUT_CSV = Path("cnj_assuntos_completos.csv")
MAX_WORKERS = 4                  # Paralelismo contido para respeitar o servidor público.
NETWORK_TIMEOUT_SECONDS = 35
NETWORK_RETRIES = 3
MIN_EXPECTED_RECORDS = 1000     # Barreira contra exportações aparentemente completas, mas vazias.

# Categorias de primeiro nível exibidas na página de assuntos do CNJ.
# Atualizadas conforme a versão da interface consultada em 12/09/2026.
ROOT_SUBJECTS: dict[int, str] = {
    12775: "DIREITO À EDUCAÇÃO",
    9985: "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO",
    10110: "DIREITO AMBIENTAL",
    12734: "DIREITO ASSISTENCIAL",
    899: "DIREITO CIVIL",
    9633: "DIREITO DA CRIANÇA E DO ADOLESCENTE",
    12480: "DIREITO DA SAÚDE",
    1156: "DIREITO DO CONSUMIDOR",
    864: "DIREITO DO TRABALHO",
    11428: "DIREITO ELEITORAL",
    10739: "DIREITO ELEITORAL E PROCESSO ELEITORAL DO STF",
    6191: "DIREITO INTERNACIONAL",
    1146: "DIREITO MARÍTIMO",
    287: "DIREITO PENAL",
    11068: "DIREITO PENAL MILITAR",
    195: "DIREITO PREVIDENCIÁRIO",
    8826: "DIREITO PROCESSUAL CIVIL E DO TRABALHO",
    1209: "DIREITO PROCESSUAL PENAL",
    11049: "DIREITO PROCESSUAL PENAL MILITAR",
    14: "DIREITO TRIBUTÁRIO",
    12467: "QUESTÕES DE ALTA COMPLEXIDADE, GRANDE IMPACTO E REPERCUSSÃO",
    7724: "REGISTROS PÚBLICOS",
}


# ================================================================
# 2. OBJETOS DE DADOS E LOGS
# ================================================================

@dataclass(frozen=True)
class Subject:
    """Representa um assunto e sua localização na árvore do SGT."""

    code: int
    description: str
    parent_code: int | None
    level: int
    status: str
    has_children: bool
    path: str


@dataclass(frozen=True)
class ChildSubject:
    """Representa a resposta mínima necessária para continuar a busca."""

    code: int
    description: str
    parent_code: int
    status: str
    has_children: bool


class SubjectGateway(Protocol):
    """Contrato isolado para permitir testes sem internet."""

    def list_children(self, parent_code: int) -> list[ChildSubject]:
        """Retorna os filhos diretos de um assunto."""
        ...

    def get_version(self) -> str:
        """Retorna a identificação da versão, quando disponível."""
        ...


class JsonLogFormatter(logging.Formatter):
    """Evita logs livres e facilita a auditoria de falhas de coleta."""

    def format(self, record: logging.LogRecord) -> str:
        return json.dumps(
            {
                "level": record.levelname,
                "event": record.getMessage(),
                "module": record.name,
            },
            ensure_ascii=False,
        )


def configure_logging() -> logging.Logger:
    """Mantém um único handler para não duplicar logs no notebook."""
    logger = logging.getLogger("cnj_assuntos")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(JsonLogFormatter())
        logger.addHandler(handler)
    logger.propagate = False
    return logger


LOGGER = configure_logging()


# ================================================================
# 3. INTEGRAÇÃO COM O WEBSERVICE OFICIAL DO CNJ
# ================================================================

SOAP_NS = "http://schemas.xmlsoap.org/soap/envelope/"
WSDL_NS = "http://schemas.xmlsoap.org/wsdl/"
SOAP_BINDING_NS = "http://schemas.xmlsoap.org/wsdl/soap/"


def get_field(value: Any, field: str) -> Any:
    """Aceita respostas de teste em objetos ou dicionários."""
    if isinstance(value, Mapping):
        return value.get(field)
    return getattr(value, field, None)


def parse_has_children(value: Any) -> bool:
    """Interpreta sem ambiguidades a indicação de descendentes."""
    if value is None or value == "":
        return False
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().upper()
    if normalized in {"1", "S", "SIM", "TRUE", "Y"}:
        return True
    if normalized in {"0", "N", "NAO", "NÃO", "FALSE", "NO"}:
        return False
    raise ValueError(f"Valor inesperado no campo 'temFilhos': {value!r}")


def local_name(tag: str) -> str:
    """Remove o namespace sem alterar o conteúdo textual do XML."""
    return tag.split("}", 1)[-1].split(":")[-1]


def find_child(element: ET.Element, name: str) -> ET.Element | None:
    """Localiza o campo pelo nome, independentemente de prefixos SOAP."""
    return next((child for child in element if local_name(child.tag) == name), None)


def resolve_reference(element: ET.Element, references: Mapping[str, ET.Element]) -> ET.Element:
    """Segue referências SOAP 'href=#id', inclusive em múltiplos níveis."""
    observed: set[str] = set()
    while True:
        href = next((str(value) for key, value in element.attrib.items()
                     if local_name(key) == "href"), None)
        if not href:
            return element
        identifier = href.lstrip("#")
        if identifier in observed or identifier not in references:
            raise ValueError(f"Referência SOAP inválida ou circular: {identifier}")
        observed.add(identifier)
        element = references[identifier]


def parse_soap_document(payload: bytes, operation: str) -> ET.Element:
    """Valida o XML SOAP e recupera a resposta da operação solicitada."""
    root = ET.fromstring(payload)
    body = root.find(f"{{{SOAP_NS}}}Body")
    if body is None:
        raise ValueError("Resposta sem SOAP Body")

    fault = find_child(body, "Fault")
    if fault is not None:
        message = find_child(fault, "faultstring")
        raise RuntimeError(
            f"CNJ retornou SOAP Fault: {message.text if message is not None else 'sem mensagem'}"
        )

    response = find_child(body, f"{operation}Response")
    if response is None:
        raise ValueError(f"Resposta SOAP sem '{operation}Response'")
    return body


def parse_soap_return(payload: bytes, operation: str) -> ET.Element | None:
    """Identifica o retorno SOAP, resolvendo arrays com multiRef."""
    body = parse_soap_document(payload, operation)
    references = {
        identifier: item
        for item in body.iter()
        for key, identifier in item.attrib.items()
        if local_name(key) == "id"
    }
    response = find_child(body, f"{operation}Response")
    assert response is not None
    value = find_child(response, "return")
    return resolve_reference(value, references) if value is not None else None


def decode_child_response(payload: bytes) -> list[ChildSubject]:
    """Converte tanto SOAP direto quanto arrays PHP SOAP/multiRef."""
    operation = "getArrayFilhosItemPublicoWS"
    body = parse_soap_document(payload, operation)
    references = {
        str(identifier): item
        for item in body.iter()
        for key, identifier in item.attrib.items()
        if local_name(key) == "id"
    }
    response = find_child(body, f"{operation}Response")
    assert response is not None
    return_element = find_child(response, "return")
    if return_element is None:
        return []
    collection = resolve_reference(return_element, references)
    if any(local_name(key) == "nil" and str(value).lower() in ("true", "1")
           for key, value in collection.attrib.items()):
        return []

    entries = [resolve_reference(child, references) for child in collection
               if local_name(child.tag) in {"item", "multiRef"}]
    if not entries and find_child(collection, "seq_elemento") is not None:
        entries = [collection]
    if not entries and len(collection) > 0:
        raise ValueError("Resposta contém estrutura desconhecida, sem elementos 'item'")

    result: list[ChildSubject] = []
    for entry in entries:
        fields = {
            local_name(field.tag): (resolve_reference(field, references).text or "").strip()
            for field in entry
        }
        if not fields.get("seq_elemento") or not fields.get("dsc_elemento"):
            raise ValueError(f"Filho retornado com código ou descrição ausente: {fields}")
        if not fields.get("seq_elemento_pai"):
            raise ValueError(f"Filho retornado sem código pai: {fields['seq_elemento']}")
        result.append(
            ChildSubject(
                code=int(fields["seq_elemento"]),
                description=fields["dsc_elemento"],
                parent_code=int(fields["seq_elemento_pai"]),
                status=fields.get("situacao", ""),
                has_children=parse_has_children(fields.get("temFilhos")),
            )
        )
    return result


class CnjSgtGateway:
    """Cliente HTTP para o SOAP antigo do CNJ, sem depender de Zeep.

    O SGT utiliza serialização SOAP herdada de versões antigas do PHP. Ler
    diretamente seu XML evita falhas conhecidas em clientes SOAP genéricos.
    """

    def __init__(
        self,
        wsdl_url: str = WSDL_URL,
        timeout: int = NETWORK_TIMEOUT_SECONDS,
        retries: int = NETWORK_RETRIES,
    ) -> None:
        if not wsdl_url.startswith("https://") or "?wsdl" not in wsdl_url:
            raise ValueError("Informe a URL HTTPS oficial do WSDL, terminada em ?wsdl.")
        self.wsdl_url = wsdl_url
        self.endpoint = wsdl_url.partition("?")[0]
        self.timeout = timeout
        self.retries = retries
        self._thread_local = threading.local()
        self._meta: dict[str, tuple[tuple[str, ...], str]] = {}

        # Lê o contrato oficial uma única vez; os workers reutilizam seus metadados.
        response = self._session().get(self.wsdl_url, timeout=self.timeout)
        response.raise_for_status()
        self._load_wsdl(response.content)

    def _session(self) -> Any:
        """Mantém conexões HTTP independentes e reutilizáveis em cada worker."""
        current = getattr(self._thread_local, "session", None)
        if current is not None:
            return current
        try:
            import requests
            from requests.adapters import HTTPAdapter
            from urllib3.util.retry import Retry
        except ImportError as exc:
            raise RuntimeError("Instale a biblioteca requests: pip install requests") from exc

        session = requests.Session()
        session.headers.update({"User-Agent": "CNJ-Assuntos-Research/1.0"})
        retry = Retry(
            total=self.retries,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=None,  # Somente operações SOAP de consulta (leitura).
            respect_retry_after_header=True,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=2, pool_maxsize=2)
        session.mount("https://", adapter)
        self._thread_local.session = session
        return session

    def _load_wsdl(self, payload: bytes) -> None:
        """Obtém nomes reais dos argumentos e SOAPAction do WSDL do CNJ."""
        root = ET.fromstring(payload)
        self.namespace = root.attrib.get("targetNamespace", "urn:sgt")
        for operation in ("getArrayFilhosItemPublicoWS", "getDataUltimaVersao"):
            request = next(
                (node for node in root.findall(f".//{{{WSDL_NS}}}message")
                 if node.attrib.get("name") == f"{operation}Request"),
                None,
            )
            if request is None:
                if operation == "getDataUltimaVersao":
                    continue
                raise ValueError(f"WSDL não declara a requisição '{operation}'")

            parts = tuple(part.attrib["name"] for part in request
                          if local_name(part.tag) == "part" and "name" in part.attrib)
            if operation == "getArrayFilhosItemPublicoWS" and len(parts) != 2:
                raise ValueError(f"Assinatura SOAP inesperada: {operation}: {parts}")
            action = f"{self.namespace}#{operation}"
            for binding in root.findall(f".//{{{WSDL_NS}}}binding"):
                for method in binding.findall(f"{{{WSDL_NS}}}operation"):
                    if method.attrib.get("name") == operation:
                        soap_info = method.find(f"{{{SOAP_BINDING_NS}}}operation")
                        if soap_info is not None:
                            action = soap_info.attrib.get("soapAction", action)
            self._meta[operation] = (parts, action)

    def _post(self, operation: str, arguments: tuple[str, ...]) -> bytes:
        """Realiza consulta SOAP conforme os nomes de parâmetros do contrato."""
        if operation not in self._meta:
            raise ValueError(f"Operação não declarada pelo WSDL: {operation}")
        parts, action = self._meta[operation]
        if len(parts) != len(arguments):
            raise ValueError(f"Argumentos incompatíveis para operação {operation}")

        envelope = ET.Element(f"{{{SOAP_NS}}}Envelope")
        body = ET.SubElement(envelope, f"{{{SOAP_NS}}}Body")
        call = ET.SubElement(body, f"{{{self.namespace}}}{operation}")
        for field_name, argument in zip(parts, arguments):
            ET.SubElement(call, f"{{{self.namespace}}}{field_name}").text = argument
        payload = ET.tostring(envelope, encoding="utf-8", xml_declaration=True)

        response = self._session().post(
            self.endpoint,
            data=payload,
            headers={"Content-Type": "text/xml; charset=utf-8", "SOAPAction": f'"{action}"'},
            timeout=self.timeout,
        )
        response.raise_for_status()
        return response.content

    def list_children(self, parent_code: int) -> list[ChildSubject]:
        """Consulta filhos de um código e verifica que os pais são coerentes."""
        payload = self._post("getArrayFilhosItemPublicoWS", (str(parent_code), "A"))
        children = decode_child_response(payload)
        for child in children:
            if child.parent_code != parent_code:
                raise ValueError(
                    f"Assunto {child.code}: pai informado {child.parent_code} "
                    f"diverge da consulta {parent_code}"
                )
        return children

    def get_version(self) -> str:
        """Consulta versão quando o método estiver publicado no WSDL."""
        if "getDataUltimaVersao" not in self._meta:
            return ""
        try:
            payload = self._post("getDataUltimaVersao", ())
            result = parse_soap_return(payload, "getDataUltimaVersao")
            return "" if result is None else "".join(result.itertext()).strip()
        except Exception as exc:
            LOGGER.warning("Versão do serviço não informada: %s", exc)
            return ""


# ================================================================
# 4. PERCURSO COMPLETO E VALIDAÇÃO DA HIERARQUIA
# ================================================================


class SubjectCollector:
    """Percorre somente nós que possuem filhos, com paralelismo controlado."""

    def __init__(
        self,
        gateway: SubjectGateway,
        roots: Mapping[int, str],
        workers: int = MAX_WORKERS,
    ) -> None:
        if not roots or workers < 1 or workers > 8:
            raise ValueError("Informe categorias raiz e entre 1 e 8 workers.")
        self.gateway = gateway
        self.roots = dict(roots)
        self.workers = workers

    def collect(self) -> dict[int, Subject]:
        """Retorna todos os nós conhecidos ou falha se algum ramo não for consultado."""
        found: dict[int, Subject] = {}
        pending: deque[Subject] = deque()

        # Todas as raízes são consultadas, inclusive quando seu status é desconhecido.
        for code, description in self.roots.items():
            if code <= 0 or not description.strip():
                raise ValueError(f"Categoria raiz inválida: {code}")
            root = Subject(code, description.strip(), None, 1, "", True, description.strip())
            found[code] = root
            pending.append(root)

        scanned = 0
        next_progress = 250
        LOGGER.info("Iniciando varredura de %d categorias de topo", len(pending))

        with ThreadPoolExecutor(max_workers=self.workers) as pool:
            while pending:
                # Limita simultaneidade e não gera milhares de requisições antecipadas.
                batch = [pending.popleft() for _ in range(min(self.workers, len(pending)))]
                futures = {pool.submit(self.gateway.list_children, node.code): node for node in batch}

                for future in as_completed(futures):
                    parent = futures[future]
                    try:
                        children = future.result()
                    except Exception as exc:
                        raise RuntimeError(
                            f"Não foi possível concluir o ramo {parent.code} - "
                            f"{parent.description}. CSV final não será gerado."
                        ) from exc
                    scanned += 1

                    if parent.has_children and parent.level > 1 and not children:
                        raise ValueError(
                            f"O servidor anunciou descendentes para {parent.code}, "
                            "mas não devolveu nenhum. Interrompendo para evitar CSV incompleto."
                        )

                    for child in children:
                        existing = found.get(child.code)
                        if existing is not None:
                            if existing.parent_code != child.parent_code:
                                raise ValueError(
                                    f"Código {child.code} apareceu em pais diferentes: "
                                    f"{existing.parent_code} e {child.parent_code}"
                                )
                            continue

                        subject = Subject(
                            code=child.code,
                            description=child.description,
                            parent_code=parent.code,
                            level=parent.level + 1,
                            status=child.status,
                            has_children=child.has_children,
                            path=f"{parent.path} > {child.description}",
                        )
                        found[subject.code] = subject
                        if subject.has_children:
                            pending.append(subject)

                if len(found) >= next_progress:
                    LOGGER.info(
                        "Andamento: %d assuntos; %d ramos consultados; %d ramos pendentes",
                        len(found), scanned, len(pending),
                    )
                    next_progress += 250

        LOGGER.info("Consulta concluída: %d assuntos, %d ramos", len(found), scanned)
        return found


# ================================================================
# 5. EXPORTAÇÃO ATÔMICA DE UM ÚNICO CSV
# ================================================================


def export_csv(subjects: Mapping[int, Subject], output: Path, version: str) -> None:
    """Grava CSV somente após validar a abrangência da extração."""
    if len(subjects) < MIN_EXPECTED_RECORDS:
        raise ValueError(
            f"Somente {len(subjects)} assuntos encontrados. "
            f"Esperados pelo menos {MIN_EXPECTED_RECORDS}: saída não gerada."
        )
    if max(subject.level for subject in subjects.values()) < 3:
        raise ValueError("A árvore parece truncada: não foram encontrados 3 níveis.")
    for code, subject in subjects.items():
        if not subject.description or (subject.parent_code is not None and subject.parent_code not in subjects):
            raise ValueError(f"Assunto inválido ou sem pai conhecido: {code}")

    output = Path(output).expanduser().resolve()
    output.parent.mkdir(parents=True, exist_ok=True)
    temp_path: Path | None = None

    try:
        # O tempfile na mesma pasta permite substituir o arquivo final atomicamente.
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8-sig", newline="", suffix=".csv.tmp",
            dir=output.parent, delete=False,
        ) as stream:
            temp_path = Path(stream.name)
            writer = csv.writer(stream, delimiter=";", quoting=csv.QUOTE_MINIMAL)
            writer.writerow(
                ["codigo", "descricao", "codigo_pai", "nivel", "situacao", "caminho", "versao_sgt"]
            )
            # Ordenação pelo caminho mantém grupos de assuntos próximos no Excel.
            for subject in sorted(subjects.values(), key=lambda row: (row.path.casefold(), row.code)):
                writer.writerow(
                    [subject.code, subject.description, subject.parent_code or "", subject.level,
                     subject.status, subject.path, version]
                )
        os.replace(temp_path, output)
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)

    digest = hashlib.sha256(output.read_bytes()).hexdigest()
    LOGGER.info("CSV gerado: %s; linhas=%d; sha256=%s", output, len(subjects), digest)


def main() -> None:
    """Ponto único de execução para Python e Jupyter Notebook."""
    gateway = CnjSgtGateway()
    version = gateway.get_version()
    LOGGER.info("Versão informada pelo SGT: %s", version or "não disponível")
    collector = SubjectCollector(gateway=gateway, roots=ROOT_SUBJECTS)
    subjects = collector.collect()
    export_csv(subjects, OUTPUT_CSV, version)
    print(f"\nConcluído: {len(subjects)} assuntos salvos em {OUTPUT_CSV.resolve()}")


if __name__ == "__main__":
    main()
